# Can a frontier LLM just do this?

**The honest version of the question.** Not an argument — a measurement.

We take pairs of Nigerian health facilities straight out of arche's crosswalk,
ask an LLM the same question arche answered, and compare. Then we look at where
each one fails and what that implies for how they should be wired together.

**Spoiler, so you read the rest with the right expectation:** the LLM is
genuinely better than arche at one thing, genuinely worse at another, and the
interesting result is *which*.

**Runtime:** about a minute, and a few cents of API credit.

---

## Setup

Put `OPENAI_API_KEY=...` in a `.env` at the repository root. The key is loaded
into the process; it is never printed.

In [1]:
import os, json, csv, collections, random, warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv("../../.env")          # repo root

MODEL = "gpt-4o-mini"              # swap for gpt-4o / o3 / claude to compare

assert os.getenv("OPENAI_API_KEY"), "no OPENAI_API_KEY found — add it to .env"
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("model:", MODEL)

API key loaded: True
model: gpt-4o-mini


## Step 1 — Get real pairs, not cherry-picked ones

The temptation in a comparison like this is to hand-pick examples that make
your tool look good. We avoid that by sampling directly from the crosswalk
output with a fixed seed: 15 pairs arche called `match`, 15 it sent to
`review`.

In [2]:
from arche.resolve import crosswalk

with open("../../data/GRID3_NGA_health_facilities_v2.csv", encoding="utf-8-sig") as fh:
    grid3 = [r for r in csv.DictReader(fh) if r["state"] == "Kano"]
with open("../../data/osm_kano.csv", encoding="utf-8-sig") as fh:
    osm = [r for r in csv.DictReader(fh) if r.get("name", "").strip()]

A = [{"name": r["name"], "lat": r["lat"], "lon": r["lon"], "type": r.get("amenity", "")} for r in osm]
B = [{"name": r["facility_name"], "lat": r["latitude"], "lon": r["longitude"],
      "type": r["facility_level_option"]} for r in grid3]

result = crosswalk(A, B, entity="place")
M = [m for m in result["matches"] if m["decision"] == "match"]
R = [m for m in result["matches"] if m["decision"] == "review"]

rng = random.Random(42)
sample = rng.sample(M, 15) + rng.sample(R, 15)
print(f"arche produced {len(M)} matches and {len(R)} reviews")
print(f"sampled {len(sample)} pairs for the comparison")

arche produced 618 matches and 289 reviews
sampled 30 pairs for the comparison


## Step 2 — Ask the model the same question

The prompt gives the model *more* than arche gets: both names, both sets of
coordinates, the distance, and the real-world stakes. If anything this is
generous.

In [3]:
from openai import OpenAI

PROMPT = """You are reconciling two Nigerian health facility registries into one Master Facility List.

Facility A: {a}   (coordinates: {alat}, {alon})
Facility B: {b}   (coordinates: {blat}, {blon})
Distance apart: {km:.2f} km

Are these the same real-world facility? A wrong merge removes a clinic from the
national list and it loses its vaccine allocation.

Reply with ONE JSON object and nothing else:
{{"verdict": "same" | "different" | "unsure", "confidence": 0.0-1.0}}"""

client = OpenAI()

def ask(a, b, km):
    p = PROMPT.format(a=a["name"], b=b["name"], alat=a["lat"], alon=a["lon"],
                      blat=b["lat"], blon=b["lon"], km=km)
    r = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=60,
                                       messages=[{"role": "user", "content": p}])
    raw = r.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    return json.loads(raw), r.usage.total_tokens

rows, tokens = [], 0
for m in sample:
    a, b = A[m["a_id"]], B[m["b_id"]]
    km = m["evidence"].get("distance_km", 0.0)
    out, t = ask(a, b, km)
    tokens += t
    rows.append({"a": a["name"], "b": b["name"], "km": km,
                 "arche": m["decision"], "score": m["score"],
                 "llm": out.get("verdict"), "conf": out.get("confidence", 0.0)})

print(f"asked {len(rows)} pairs, {tokens:,} tokens")

asked 30 pairs, 4,927 tokens


## Step 3 — Where they agree and disagree

In [4]:
xt = collections.Counter((r["arche"], r["llm"]) for r in rows)
print(f"{'arche':10} {'LLM':12} {'n':>4}")
print("-" * 28)
for (ar, lm), n in sorted(xt.items()):
    print(f"{ar:10} {lm:12} {n:>4}")

rev = [r for r in rows if r["arche"] == "review"]
print()
print(f"On the {len(rev)} pairs arche sent to REVIEW:")
print(f"   LLM also hedged ('unsure')     : {sum(1 for r in rev if r['llm']=='unsure')}")
print(f"   LLM committed at conf >= 0.7   : {sum(1 for r in rev if r['conf']>=0.7)}")

arche      LLM             n
----------------------------
match      different       5
match      same            7
match      unsure          3
review     different      11
review     same            4

On the 15 pairs arche sent to REVIEW:
   LLM also hedged ('unsure')     : 0
   LLM committed at conf >= 0.7   : 15


**The model never hedges.** On every pair arche declined to decide, the LLM
returned a confident verdict. That is not a bug in the model — it is what
"helpful assistant" means. It is, however, exactly the property you do not want
in the component that decides whether two clinics are one clinic.

But confident is not the same as wrong. Let's look at *what* it committed to.

## Step 4 — Where the LLM is genuinely better

These are pairs arche refused and the model merged. Read the names.

In [5]:
for r in rows:
    if r["arche"] == "review" and r["llm"] == "same":
        print(f'  conf {r["conf"]:.2f}  {r["a"][:40]:42} == {r["b"][:40]:42} ({r["km"]:.2f} km)')

  conf 0.95  Sarigarin Health Post                      == Sari Girin Health Post                     (0.00 km)
  conf 0.95  Maitsidau Primary Health Centre            == Mai Tsidau Primary Health Center           (0.00 km)
  conf 0.95  Tsalle Health Post                         == Tsalle Primary Health Care Center          (0.00 km)
  conf 0.95  Ririwai Primary Health Centre              == Riruwai Primary Health Center              (0.00 km)


These are **word-boundary variants in Hausa names** — the same name written
with or without a space. At identical coordinates, these are almost certainly
the same facility, and the model is right about them.

arche's name comparator treats a space as a token boundary, so `Sarigarin` and
`Sari Girin` share no tokens and the distinctiveness gate never fires. **This
is a real gap in arche, found by running this comparison.** It is a data
problem, not an architecture problem: the fix is a Hausa orthography rule in
the place pack, not a different engine.

Credit where it is due — this is exactly the kind of variation a language model
handles well and a token-based comparator does not.

## Step 5 — Where the LLM fails badly

Now the other direction. These are pairs arche matched confidently and the
model did not.

In [6]:
for r in rows:
    if r["arche"] == "match" and r["llm"] != "same":
        same_name = r["a"].strip().lower() == r["b"].strip().lower()
        flag = "  <-- IDENTICAL NAME" if same_name else ""
        print(f'  arche {r["score"]:.3f} | LLM {r["llm"]:9} conf {r["conf"]:.2f} | '
              f'{r["a"][:34]:36} vs {r["b"][:34]:36} ({r["km"]:.2f} km){flag}')

  arche 0.998 | LLM unsure    conf 0.70 | Yanchibi Health Post                 vs Yanchibi Health Post                 (0.02 km)  <-- IDENTICAL NAME
  arche 0.999 | LLM unsure    conf 0.70 | Agalawa Health Post                  vs Agalawa Health Post                  (0.01 km)  <-- IDENTICAL NAME
  arche 1.000 | LLM unsure    conf 0.80 | Kurugu Health Post                   vs Kurugu Health Post                   (0.00 km)  <-- IDENTICAL NAME
  arche 0.747 | LLM different conf 0.90 | Durba Health Post                    vs Durba Primary Health Center          (0.04 km)
  arche 1.000 | LLM different conf 0.90 | Alfindi Health Post                  vs Alfindi Health Post                  (0.00 km)  <-- IDENTICAL NAME
  arche 0.715 | LLM different conf 0.90 | Leni Health Post                     vs Leni Basic Health Center             (0.00 km)
  arche 0.804 | LLM different conf 0.95 | Tukuibi Health Post                  vs Tukuibi Health Post                  (11.48 km)  <-- IDENTICAL N

**Look at the flagged rows.** Identical facility names, at coordinates
metres apart, and the model returned `different` or `unsure` — one of them at
0.90 confidence.

There is no reading of the evidence under which those are different facilities.
The model invented a distinction because it was asked a question and producing
an answer is what it does. Nothing in its output signals that this verdict is
less reliable than any other; the confidence score says 0.90 either way.

One row is worth defending on the model's side: a pair with an identical name
**11.5 km apart**, which arche matched and the model called different. The
model is probably right — that is likely two villages sharing a name. arche's
geographic decay was too forgiving there.

So: **both make mistakes, and they make different mistakes.**

## Step 6 — The property that decides the architecture

Accuracy is not the only axis. Ask the same question five times and see what
happens.

In [7]:
probe_a = {"name": "Gurduba Health Post", "lat": "11.7", "lon": "8.5"}
probe_b = {"name": "Gurduba Primary Health Care", "lat": "11.7", "lon": "8.5"}

print("Same question, five times, temperature=0:")
answers = []
for i in range(5):
    out, _ = ask(probe_a, probe_b, 0.38)
    answers.append((out.get("verdict"), out.get("confidence")))
    print(f"   run {i+1}: {out}")

print()
print("distinct answers:", len(set(answers)))
print()
from arche import resolve
ids = {resolve.pairwise("Gurduba Health Post", "Gurduba Primary Health Care").decision_id
       for _ in range(5)}
print("arche, five runs, distinct decision_ids:", len(ids))

Same question, five times, temperature=0:


   run 1: {'verdict': 'same', 'confidence': 0.9}


   run 2: {'verdict': 'unsure', 'confidence': 0.7}


   run 3: {'verdict': 'same', 'confidence': 0.9}


   run 4: {'verdict': 'same', 'confidence': 0.9}


   run 5: {'verdict': 'unsure', 'confidence': 0.7}

distinct answers: 2



Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


arche, five runs, distinct decision_ids: 1


Even at `temperature=0`, a hosted model is not a stable function: the
serving stack, the model version behind the alias, and batching all move
underneath you. arche returns the same content-addressed `decision_id` every
time.

That matters because **you cannot attest what you cannot replay.** A compliance
artifact whose answer changes between runs is not an artifact. This is not a
quality gap that a better model closes — it is a different category of
guarantee.

## Step 7 — What this actually implies: route the review queue

The measurement points at an architecture rather than a winner.

| | arche | LLM |
|---|---|---|
| Word-boundary and transliteration variants | misses some | **strong** |
| Identical names at identical coordinates | correct | **unreliable** |
| Abstains when evidence is thin | **by construction** | never |
| Same answer twice | **guaranteed** | no |
| Cost per pair at national scale | ~0 | real money |

Neither should be the whole system. The sensible wiring:

1. **arche blocks and gates.** It reduces 1.2M possible pairs to 40k scored
   ones, resolves the unambiguous majority deterministically, and refuses the
   rest. Nothing expensive has happened yet.
2. **The review queue routes to the LLM.** That is ~100 pairs, not 40,000 —
   the model is asked only about cases where a human would otherwise be needed,
   and its recall on name variants is exactly what is useful there.
3. **The model proposes; it never merges.** Its verdict enters as evidence with
   a measured reliability, not as a decision.
4. **A human confirms**, and the adjudication is signed.

The cost difference is the whole argument for that ordering. Let's price it.

In [8]:
per_pair = tokens / len(rows)
print(f"measured: {per_pair:.0f} tokens per pair")

TOTAL_PAIRS = 138_232_282     # 51,022 GRID3 records x a second national list, after blocking
REVIEW_ONLY = int(TOTAL_PAIRS * 0.146)   # the review share we measured in Kano

for label, usd_per_m in [("gpt-4o-mini", 0.15), ("mid-tier", 3.0), ("frontier", 15.0)]:
    everything = TOTAL_PAIRS * per_pair / 1_000_000 * usd_per_m
    queue_only = REVIEW_ONLY  * per_pair / 1_000_000 * usd_per_m
    print(f"{label:12}  LLM on every pair: ${everything:>12,.0f}   "
          f"LLM on the review queue only: ${queue_only:>11,.0f}")

measured: 164 tokens per pair
gpt-4o-mini   LLM on every pair: $       3,405   LLM on the review queue only: $        497
mid-tier      LLM on every pair: $      68,107   LLM on the review queue only: $      9,944
frontier      LLM on every pair: $     340,535   LLM on the review queue only: $     49,718


Running the model over everything is how you turn a laptop job into a
line item. Running it over the review queue only is a rounding error — and it
is applied precisely where it adds recall.

**That is the product**: a deterministic gate that makes the expensive,
non-replayable component cheap to use, and honest about having been used.

## Step 8 — Doing it properly, inside arche

You would not wire this by hand in production. arche's LLM lane already has the
right shape — the model fills a schema *you* declare, hallucinated fields
become violations rather than values, and the deterministic engine grades the
model's judgment against itself:

```python
from arche.llm.harness import grade_pairs

report = grade_pairs(decl, review_queue, judge=my_llm_judge)
report.agreement_rate      # None if nothing was scorable
report.engine_abstained    # pairs where the engine declined — not the model's fault
report.divergences         # every disagreement, WITH the engine's evidence
```

Engine `review` outcomes count as **abstentions**, not misses. The judge's
vocabulary is closed — returning `"probably"` raises. And every divergence
carries the factor-by-factor evidence, so "the model disagreed" is always
inspectable.

Then the decision is signed with an honest account of the model's role:

```python
decision.pins["extraction"]
# {"model": "gpt-4o-mini", "reproducible": False, ...}
```

`reproducible: False`, because a hosted model was involved. The representation
and the decision maths replay; the extraction does not. The signature covers
the split rather than papering over it.

See **[Bring your own LLM](../../docs-site/docs/how-to/bring-your-own-llm.md)**.

## What this notebook found

Running it, rather than arguing about it, produced three things we did not know
going in:

1. **A real gap in arche.** Hausa word-boundary variants (`Sarigarin` /
   `Sari Girin`) are missed by the token comparator. That is a data fix for the
   place pack, and it is now a known issue rather than a guess.
2. **A real failure mode in the model.** It returned `different` at 0.90
   confidence for two records with an identical name at identical coordinates.
   Confidence carried no signal about that.
3. **The architecture.** Neither component should be the whole system. arche
   gates cheaply and refuses honestly; the model adds recall on the small
   residue; a human confirms; the signature records who did what.

The claim worth making is not "we beat the model". It is: **we make the model
affordable to use where it helps, and we keep it away from the decision it
cannot be trusted with.**

## Try it yourself

- Change `MODEL` to `gpt-4o` or `o3` — does the identical-name failure persist?
- Raise the sample size and see whether the crosstab holds
- Feed the model the facility *type* explicitly and see if the tier-difference
  cases (`Health Post` vs `Primary Health Care Centre`) improve